# 01. Разведочный анализ данных HiggsML

**Цель ноутбука:** понять, что лежит в наборе данных, прежде чем обучать модели.
После него вы должны уметь ответить на вопросы комиссии: «сколько у вас сигнала?»,
«что такое веса?», «почему в данных −999?» и «какие признаки разделяют сигнал и фон?».

**Данные:** ATLAS Higgs Boson Machine Learning Challenge 2014, CERN Open Data, запись 328,
DOI 10.7483/OPENDATA.ATLAS.ZBP2.M5T8 (данные) и 10.7483/OPENDATA.ATLAS.MQ5J.GHXA (документация).
Сигнал — распад бозона Хиггса H → ττ (один τ распадается в лептон, другой — в адроны),
фон — Z → ττ, t t̄, W + струи. Все события — Monte-Carlo симуляция детектора ATLAS, 8 ТэВ, 2012 г.

Цитировать как: Adam-Bourdarios C. et al., *The Higgs boson machine learning challenge*, PMLR 42:19–55, 2015.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Корень репозитория: поднимаемся, пока не найдём каталог src/
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import (load_higgsml, find_data_file, FEATURES, DER_FEATURES, PRI_FEATURES,
                      FEATURE_LABELS, MISSING, with_nan, missing_mask, expected_yields,
                      split_train_val_test)
from src.metrics import (ams, z0, weighted_counts, weighted_roc, significance_scan, best_threshold,
                         evaluate_at_threshold, ks_overtraining, balanced_weights)
from src.plots import setup_style, hist_sig_bkg, grid_all_features, plot_roc, plot_significance_scan, plot_classifier_output

setup_style()
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
DATA_PATH = find_data_file(ROOT / "data")
print("Файл данных:", DATA_PATH.name)
if "synthetic" in DATA_PATH.name:
    print("ВНИМАНИЕ: это синтетические данные для проверки кода, а не физика!")

In [ ]:
df = load_higgsml(DATA_PATH)
print(f"Событий: {len(df):,}, столбцов: {df.shape[1]}")
df.head()

## 1. Сколько сигнала на самом деле

В таблице треть строк — сигнал. Но это обман: симуляцию сигнала генерируют
«с запасом», чтобы хорошо описать его форму, а вес каждого события приводит
число к тому, что ожидается в реальных данных 2012 года. Поэтому важны **суммы весов**.

In [ ]:
yields = expected_yields(df)
rows = pd.DataFrame({
    "строк в таблице": [yields["n_signal_rows"], yields["n_background_rows"]],
    "ожидается событий (сумма весов)": [yields["s"], yields["b"]],
}, index=["сигнал (s)", "фон (b)"])
rows["доля"] = rows.iloc[:, 1] / rows.iloc[:, 1].sum()
display(rows.round(3))
print(f"Отношение s/b = {yields['s'] / yields['b']:.5f};  s/sqrt(b) без всякого отбора = {yields['s'] / np.sqrt(yields['b']):.3f}")
print(f"Значимость без отбора: Z0 = {float(z0(yields['s'], yields['b'])):.3f},  AMS = {float(ams(yields['s'], yields['b'])):.3f}")

Вывод для главы 2: задача — не «угадать класс», а **найти область фазового пространства,
где отношение s/√b максимально**. Именно поэтому главная метрика работы — значимость,
а не accuracy.

## 2. Структура значений −999

−999 — это не пропуск по ошибке, а «величина не определена». Если в событии нет второй
струи, то нет и массы пары струй. Масса MMC не определена, когда алгоритм восстановления
массы ττ не сходится. Посмотрим, как это связано с числом струй.

In [ ]:
mm = missing_mask(df)
frac = mm.mean().sort_values(ascending=False)
frac = frac[frac > 0]
display(frac.rename("доля −999").to_frame().T.round(3))

# Доля −999 по каждому признаку в зависимости от числа струй
by_jets = mm.groupby(df["PRI_jet_num"]).mean()
by_jets = by_jets.loc[:, (by_jets > 0).any()]
display(by_jets.round(2))

Вывод: группы признаков «включаются» при PRI_jet_num ≥ 1 и ≥ 2. Для моделей это значит:
- деревья (XGBoost/LightGBM) умеют работать с NaN напрямую — заменяем −999 на NaN;
- для логистической регрессии и нейросети нужны импутация **и** бинарный флаг «было −999»,
  иначе модель примет −999 за очень маленькую массу.

## 3. Распределения признаков: сигнал против фона

Все гистограммы взвешенные и нормированы на единицу — сравниваем **форму**.

In [ ]:
fig = grid_all_features(df, ncols=5, save_to=RESULTS / "fig_eda_all_features.png")
plt.show()

## 4. Ключевые признаки крупно

Физически ожидаем: масса ττ (MMC) у сигнала около 125 ГэВ, у фона Z → ττ — около 91 ГэВ;
поперечная масса m_T(MET, ℓ) мала у сигнала и велика у W + струи; ΔR(τ, ℓ) меньше у сигнала,
так как бозон Хиггса тяжелее Z и τ-лептоны летят ближе друг к другу при бусте.

In [ ]:
key = ["DER_mass_MMC", "DER_mass_transverse_met_lep", "DER_mass_vis",
       "PRI_met", "DER_deltar_tau_lep", "DER_pt_ratio_lep_tau"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), key):
    hist_sig_bkg(df, col, ax=ax, bins=50)
fig.tight_layout()
fig.savefig(RESULTS / "fig_eda_key_features.png", bbox_inches="tight")
plt.show()

## 5. Корреляции признаков

Сильно коррелированные признаки (|ρ| > 0.9) не добавляют информации линейной модели
и мешают интерпретации важности. Считаем отдельно для сигнала и фона: корреляции могут
различаться, и это тоже разделяющая информация.

In [ ]:
dn = with_nan(df)
corr_s = dn.loc[dn["y"] == 1, FEATURES].corr()
corr_b = dn.loc[dn["y"] == 0, FEATURES].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, corr, name in zip(axes, (corr_s, corr_b), ("сигнал", "фон")):
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(FEATURES))); ax.set_xticklabels(FEATURES, rotation=90, fontsize=7)
    ax.set_yticks(range(len(FEATURES))); ax.set_yticklabels(FEATURES, fontsize=7)
    ax.set_title(f"корреляции Пирсона, {name}")
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.6)
fig.savefig(RESULTS / "fig_eda_correlations.png", bbox_inches="tight")
plt.show()

pairs = (corr_s.where(np.triu(np.ones(corr_s.shape), 1).astype(bool)).stack()
         .rename("rho_signal").to_frame())
pairs["rho_background"] = corr_b.stack().reindex(pairs.index)
print("Пары с |rho| > 0.8 (сигнал):")
display(pairs[pairs["rho_signal"].abs() > 0.8].round(2))

## 6. Разделяющая способность каждого признака по отдельности

Берём значение признака как «выход классификатора» и считаем взвешенный ROC AUC.
AUC = 0.5 — признак бесполезен сам по себе; чем дальше от 0.5 — тем лучше.
Это первое приближение к важности признаков; окончательную важность даст SHAP на этапе 4.

In [ ]:
aucs = {}
for col in FEATURES:
    m = df[col] != MISSING
    if m.sum() < 100 or df.loc[m, "y"].nunique() < 2:
        continue
    _, _, auc = weighted_roc(df.loc[m, "y"], df.loc[m, col], df.loc[m, "Weight"])
    aucs[col] = max(auc, 1 - auc)
auc_s = pd.Series(aucs).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(auc_s.index[::-1], auc_s.values[::-1], color="#4c72b0")
ax.axvline(0.5, color="k", lw=0.8)
ax.set_xlabel("взвешенный ROC AUC одного признака")
ax.set_xlim(0.45, max(0.8, auc_s.max() + 0.02))
fig.tight_layout()
fig.savefig(RESULTS / "fig_eda_single_feature_auc.png", bbox_inches="tight")
plt.show()
display(auc_s.head(10).round(3).rename("AUC").to_frame().T)

## 7. Первая значимость без машинного обучения: окно по массе

Самый простой «анализ»: оставить события в окне масс MMC вокруг 125 ГэВ.
Сканируем ширину окна и смотрим, какой AMS это даёт. Это число станет первой
строкой таблицы результатов (пороговый отбор по одному признаку).

In [ ]:
mass = df["DER_mass_MMC"].to_numpy()
ok = mass != MISSING
y = df["y"].to_numpy(); w = df["Weight"].to_numpy()

records = []
for lo in range(80, 125, 5):
    for hi in range(126, 180, 5):
        sel = ok & (mass >= lo) & (mass <= hi)
        s, b = weighted_counts(y, sel, w)
        records.append({"m_lo": lo, "m_hi": hi, "s": s, "b": b, "AMS": float(ams(s, b)), "Z0": float(z0(s, b))})
scan = pd.DataFrame(records).sort_values("AMS", ascending=False)
best = scan.iloc[0]
print(f"Лучшее окно: {best.m_lo:.0f} < m_MMC < {best.m_hi:.0f} ГэВ  →  s = {best.s:.1f}, b = {best.b:.0f}, AMS = {best.AMS:.3f}")
display(scan.head(5).round(3))

## Выводы и что записать в главу 2

1. Доли сигнала по строкам и по весам (таблица из п. 1) — с объяснением, что такое вес.
2. Таблица структуры −999 по числу струй (п. 2) и принятая стратегия обработки.
3. Рисунок с 6 ключевыми признаками (п. 4) и физическое объяснение различий.
4. Ранжирование признаков по одиночному AUC (п. 6) — «ожидаем, что эти признаки будут главными».
5. AMS простого окна по массе (п. 7) — точка отсчёта для всех моделей.

Следующий ноутбук: `02_baseline.ipynb` — пороговый отбор по нескольким признакам,
логистическая регрессия, превью XGBoost.

In [ ]:
summary = {
    "data_file": DATA_PATH.name,
    "n_events": int(len(df)),
    "expected_yields": yields,
    "best_mass_window": {k: float(v) for k, v in best.to_dict().items()},
    "top_single_feature_auc": auc_s.head(10).round(4).to_dict(),
}
(RESULTS / "eda_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print("Сохранено:", RESULTS / "eda_summary.json")